# Домашнее задание 4

Реализуйте поиск объекта с референсного изображения на кадрах видео с помощью поиска и сопоставления ключевых точек. В этом задании рекомендуется активно пользоваться гуглом :)

Скорее всего сопоставление не будет идеальным. Этого и не требуется, достаточно продемонстрировать работоспособность как минимум на первой половине видео.

Объясните в отдельной клетке, какие эффекты наблюдаются ближе к концу видео и почему.

In [84]:
pip install opencv-contrib-python>=4.4

Note: you may need to restart the kernel to use updated packages.


In [85]:
import numpy as np
import cv2

ref_path = "ref.jpg"
vid_path = "vid.mp4"

img = cv2.imread(ref_path)
ref = cv2.imread(ref_path, cv2.IMREAD_GRAYSCALE)

top = 620
bottom = 660
left = 350
right = 295

height, width = ref.shape[:2]
ref = ref[top:height-bottom, left:width-right]

sift = cv2.SIFT_create()

kp_ref, desc_ref = sift.detectAndCompute(ref, None)

bf = cv2.BFMatcher()

cam = cv2.VideoCapture(vid_path)

while (True):
    success, frame = cam.read()
    if(success == False):
        cam.release()
        cam = cv2.VideoCapture(vid_path)  
        continue
    
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    
    kp_vid, desc_vid = sift.detectAndCompute(frame, None)
    
    matches = bf.knnMatch(desc_ref, desc_vid, k=2)
    
    good_matches = []
    points = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)
            points.append(kp_vid[m.trainIdx].pt)
            
    img_matches = cv2.drawMatches(ref, kp_ref, 
                                  frame, kp_vid, 
                                  good_matches, None,
                                  flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    
    if len(points) >= 4:
        points_array = np.array(points, dtype=np.float32)

        # Сдвиг x-координаты
        points_array[:, 0] += ref.shape[1]

        # Фильтрация выбросов
        center = np.mean(points_array, axis=0)
        dists = np.linalg.norm(points_array - center, axis=1)
        mean_dist = np.mean(dists)
        std_dist = np.std(dists)

        # Только те, которые ближе 2 стандартных отклонения
        mask = dists < (mean_dist + 3 * std_dist)
        filtered_points = points_array[mask]

        if len(filtered_points) > 0:
            x, y, w, h = cv2.boundingRect(filtered_points)

            pad = 10
            x = max(0, x - pad)
            y = max(0, y - pad)
            w = min(img_matches.shape[1] - x, w + 2 * pad)
            h = min(img_matches.shape[0] - y, h + 2 * pad)

            cv2.rectangle(img_matches, (x, y), (x + w, y + h), (0, 255, 0), 3)
    
    cv2.namedWindow('matches', cv2.WINDOW_KEEPRATIO)
    cv2.imshow('matches', img_matches)
    cv2.resizeWindow('matches', 1280, 720)
    
    
    key = cv2.waitKey(30) & 0xFF
    if key == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()
cv2.waitKey(10)
    

-1

# Что происходит во второй половине видоса?

В задании я использовал SIFT. Он ищет ключевые точки по "пирамиде разностей градиентов". В конце записи, как можно заметить, алгос отрабатывает так себе: точки струбцины сопоставляются с точками, например, стула, который стоит на фоне. Думаю, это объяснимо:
 1) Струбцина уже не главный объект и в кадре она занимает меньше места => меньше пикселей, SIFT с таким может плохо отрабатывать. 
 
 2) Вносит хаос, изменение освещение из-за автофокусироваки камеры, которая стремится показать на картинке ВСЕ хорошо, шум (оказывает больше влияния, когда струбцина меньше в кадре), движение и размытие кадра (картинка не статичная).
 
 В итоге, дескрипторы струбцины и стула получаются очень близки. И алгоритм воспринимает их как одно и то же.

Вывод: если речь идет о задаче детекции со статичными фреймами, SIFT - крутая штука.